In [ ]:
from google.colab import files
uploaded = files.upload()

Saving dataset_export.csv to dataset_export.csv


In [ ]:
import pandas as pd
import numpy as np
import os
import json
import joblib
from tabulate import tabulate
import ast


from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, confusion_matrix


# DATA_PATH = "DSL-StrongPasswordData.csv"
DATA_PATH = "dataset_export.csv"
MODEL_DIR = "models"

THRESHOLD_FILE = "thresholds.json"
METRICS_FILE = "metrics.json"

os.makedirs(MODEL_DIR, exist_ok=True)

# LOAD DATASET STRONGPASSWORD

In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.columns)

Index(['subject_code', 'name_initial', 'device_info', 'repetition', 'H_vector',
       'DD_vector', 'UD_vector', 'UU_vector', 'DU_vector', 'H_mean', 'H_std',
       'H_min', 'H_max', 'H_cv', 'DD_mean', 'DD_std', 'DD_min', 'DD_max',
       'DD_cv', 'UD_mean', 'UD_std', 'UD_min', 'UD_max', 'UD_cv', 'UU_mean',
       'UU_std', 'UU_min', 'UU_max', 'UU_cv', 'DU_mean', 'DU_std', 'DU_min',
       'DU_max', 'DU_cv', 'total_duration', 'typing_speed', 'created_at'],
      dtype='object')


In [ ]:
subject_counts = df["subject_code"].value_counts()
valid_users = subject_counts[subject_counts >= 100].index

df_filtered = df[df["subject_code"].isin(valid_users)].reset_index(drop=True)

print("Total user sebelum filter:", df["subject_code"].nunique())
print("Total user setelah filter:", df_filtered["subject_code"].nunique())

DROP_COLUMNS = ["subject_code", "name_initial", "device_info", "repetition","H_vector","DD_vector","UD_vector","UU_vector","DU_vector", "created_at"]

X_all = df_filtered.drop(columns=DROP_COLUMNS)
subjects = df_filtered["subject_code"]
users = subjects.unique()
print("Total users:", len(users))

Total user sebelum filter: 12
Total user setelah filter: 10
Total users: 10


LOAD DATASET KAMI

In [ ]:
print(X_all.shape)

(1000, 27)


COMPARISON METRIC FUNCTION

In [ ]:
def compute_metrics(y_true, probs, threshold):

    y_pred = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    FAR = fp / (fp + tn) if (fp + tn) != 0 else 0
    FRR = fn / (fn + tp) if (fn + tp) != 0 else 0

    return {
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "accuracy": accuracy,
        "FAR": FAR,
        "FRR": FRR
    }


def compute_eer_threshold(y_true, probs):
    fpr, tpr, thresholds = roc_curve(y_true, probs)
    fnr = 1 - tpr
    idx = np.argmin(np.abs(fpr - fnr))
    eer = (fpr[idx] + fnr[idx]) / 2
    threshold = thresholds[idx]
    return eer, threshold

def evaluate_model(model, X, y, threshold, dataset_name="TEST"):
    probs = model.predict_proba(X)[:, 1]
    metrics = compute_metrics(y, probs, threshold)
    eer, _ = compute_eer_threshold(y, probs)

    return metrics, eer

In [ ]:
def print_split_distribution(user, y_train, y_val, y_test):
    train_genuine = np.sum(y_train == 1)
    train_impostor = np.sum(y_train == 0)

    val_genuine = np.sum(y_val == 1)
    val_impostor = np.sum(y_val == 0)

    test_genuine = np.sum(y_test == 1)
    test_impostor = np.sum(y_test == 0)

    print(f"\nDistribusi untuk user {user}:")
    print(f"TRAIN      -> Genuine: {train_genuine}, Impostor: {train_impostor}, Total: {len(y_train)}")
    print(f"VALIDATION -> Genuine: {val_genuine}, Impostor: {val_impostor}, Total: {len(y_val)}")
    print(f"TEST       -> Genuine: {test_genuine}, Impostor: {test_impostor}, Total: {len(y_test)}")

PARAMETER GRID

In [ ]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10],
    "min_samples_leaf": [1, 2, 3],
    "max_features": ["sqrt"]
}

KODE TAMBAHAN UNTUK .CSV DATASET BARYU

TRAINING LOOP UTAMA

In [ ]:
confusion_records = []
accuracy_records = []
thresholds = {}
metrics_all = {}

for user in users:
    print("\nTraining user:", user)
    y = (subjects == user).astype(int)

    X_train, X_temp, y_train, y_temp = train_test_split(
        X_all, y,
        test_size=0.4,
        stratify=y,
        random_state=42
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=0.5,
        stratify=y_temp,
        random_state=42
    )
    print_split_distribution(user, y_train, y_val, y_test)
    best_model = None
    best_eer = 1.0


    for n in param_grid["n_estimators"]:
        for depth in param_grid["max_depth"]:
            for leaf in param_grid["min_samples_leaf"]:
                for feat in param_grid["max_features"]:

                    model = RandomForestClassifier(
                        n_estimators=n,
                        max_depth=depth,
                        min_samples_leaf=leaf,
                        max_features=feat,
                        class_weight="balanced",
                        random_state=42,
                        n_jobs=-1
                    )

                    model.fit(X_train, y_train)
                    val_probs = model.predict_proba(X_val)[:,1]
                    eer, threshold = compute_eer_threshold(y_val, val_probs)

                    if eer < best_eer:
                        best_eer = eer
                        best_model = model
                        best_threshold = threshold


    print("Best EER:", best_eer)
    threshold = best_threshold
    thresholds[user] = best_threshold

    train_metrics, train_eer = evaluate_model(
        best_model, X_train, y_train, threshold, "TRAIN"
    )

    test_metrics, test_eer = evaluate_model(
        best_model, X_test, y_test, threshold, "TEST"
    )
    # --- Confusion Matrix TRAIN ---
    confusion_records.append({
        "user": user,
        "dataset": "TRAIN",
        "TP": train_metrics["TP"],
        "TN": train_metrics["TN"],
        "FP": train_metrics["FP"],
        "FN": train_metrics["FN"]
    })

    # --- Confusion Matrix TEST ---
    confusion_records.append({
        "user": user,
        "dataset": "TEST",
        "TP": test_metrics["TP"],
        "TN": test_metrics["TN"],
        "FP": test_metrics["FP"],
        "FN": test_metrics["FN"]
    })

    # --- Accuracy Metrics TRAIN ---
    accuracy_records.append({
        "user": user,
        "dataset": "TRAIN",
        "accuracy": train_metrics["accuracy"],
        "FAR": train_metrics["FAR"],
        "FRR": train_metrics["FRR"],
        "EER": train_eer
    })

    # --- Accuracy Metrics TEST ---
    accuracy_records.append({
        "user": user,
        "dataset": "TEST",
        "accuracy": test_metrics["accuracy"],
        "FAR": test_metrics["FAR"],
        "FRR": test_metrics["FRR"],
        "EER": test_eer
    })
    joblib.dump(best_model, f"models/{user}.pkl")


Training user: s003

Distribusi untuk user s003:
TRAIN      -> Genuine: 60, Impostor: 540, Total: 600
VALIDATION -> Genuine: 20, Impostor: 180, Total: 200
TEST       -> Genuine: 20, Impostor: 180, Total: 200
Best EER: 0.0

Training user: s004

Distribusi untuk user s004:
TRAIN      -> Genuine: 60, Impostor: 540, Total: 600
VALIDATION -> Genuine: 20, Impostor: 180, Total: 200
TEST       -> Genuine: 20, Impostor: 180, Total: 200
Best EER: 0.06388888888888891

Training user: s005

Distribusi untuk user s005:
TRAIN      -> Genuine: 60, Impostor: 540, Total: 600
VALIDATION -> Genuine: 20, Impostor: 180, Total: 200
TEST       -> Genuine: 20, Impostor: 180, Total: 200
Best EER: 0.050000000000000024

Training user: s009

Distribusi untuk user s009:
TRAIN      -> Genuine: 60, Impostor: 540, Total: 600
VALIDATION -> Genuine: 20, Impostor: 180, Total: 200
TEST       -> Genuine: 20, Impostor: 180, Total: 200
Best EER: 0.005555555555555556

Training user: s011

Distribusi untuk user s011:
TRAIN   

SAVE RESULT

In [ ]:

# BUAT DATAFRAME
confusion_df = pd.DataFrame(confusion_records)
accuracy_df = pd.DataFrame(accuracy_records)

# Bulatkan metric agar rapi
accuracy_df[["accuracy", "FAR", "FRR", "EER"]] = accuracy_df[["accuracy", "FAR", "FRR", "EER"]].round(5)

# SIMPAN FILE
confusion_df.to_csv("Confusion_Metric_File.csv", index=False)
joblib.dump(confusion_df, "Confusion_Metric_File.joblib")

accuracy_df.to_csv("Accuration_Metric_File.csv", index=False)
joblib.dump(accuracy_df, "Accuration_Metric_File.joblib")

# PRINT TABEL
print("\n===== CONFUSION METRIC TABLE =====")
print(tabulate(confusion_df, headers="keys", tablefmt="grid", showindex=False))

print("\n===== ACCURATION METRIC TABLE =====")
print(tabulate(accuracy_df, headers="keys", tablefmt="grid", showindex=False))

# HITUNG RATA-RATA TEST DAN TRAIN


test_only = accuracy_df[accuracy_df["dataset"] == "TEST"]

avg_metrics = {
    "Average Accuracy": test_only["accuracy"].mean(),
    "Average FAR": test_only["FAR"].mean(),
    "Average FRR": test_only["FRR"].mean(),
    "Average EER": test_only["EER"].mean()
}

avg_df = pd.DataFrame([avg_metrics]).round(4)

print("\n===== AVERAGE SYSTEM PERFORMANCE (TEST) =====")
print(tabulate(avg_df, headers="keys", tablefmt="grid", showindex=False))

test_only = accuracy_df[accuracy_df["dataset"] == "TRAIN"]

avg_metrics = {
    "Average Accuracy": test_only["accuracy"].mean(),
    "Average FAR": test_only["FAR"].mean(),
    "Average FRR": test_only["FRR"].mean(),
    "Average EER": test_only["EER"].mean()
}

avg_df = pd.DataFrame([avg_metrics]).round(4)

print("\n===== AVERAGE SYSTEM PERFORMANCE (TRAIN) =====")
print(tabulate(avg_df, headers="keys", tablefmt="grid", showindex=False))


===== CONFUSION METRIC TABLE =====
+--------+-----------+------+------+------+------+
| user   | dataset   |   TP |   TN |   FP |   FN |
+========+===========+======+======+======+======+
| s003   | TRAIN     |   60 |  539 |    1 |    0 |
+--------+-----------+------+------+------+------+
| s003   | TEST      |   18 |  180 |    0 |    2 |
+--------+-----------+------+------+------+------+
| s004   | TRAIN     |   60 |  521 |   19 |    0 |
+--------+-----------+------+------+------+------+
| s004   | TEST      |   20 |  165 |   15 |    0 |
+--------+-----------+------+------+------+------+
| s005   | TRAIN     |   60 |  520 |   20 |    0 |
+--------+-----------+------+------+------+------+
| s005   | TEST      |   20 |  167 |   13 |    0 |
+--------+-----------+------+------+------+------+
| s009   | TRAIN     |   60 |  538 |    2 |    0 |
+--------+-----------+------+------+------+------+
| s009   | TEST      |   20 |  177 |    3 |    0 |
+--------+-----------+------+------+------+---

In [ ]:
with open("thresholds.json", "w") as f:
    json.dump(thresholds, f, indent=4)

print("Done.")

Done.
